In [1]:
import os

from pathlib import Path

current_dir = Path(os.getcwd())
print(f"Current directory: {current_dir}")
for parent in current_dir.parents:
    if "pyproject.toml" in os.listdir(parent):
        print(f"Found pyproject.toml in: {parent}")
        os.chdir(parent)
        break
 

Current directory: /home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/experiments
Found pyproject.toml in: /home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows


In [2]:
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.redis import AsyncRedisSaver
from pprint import pprint   

   
from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import (
    REDIS_URL,
    install_safe_pending_sends_loader,
)

install_safe_pending_sends_loader()

MODE = "own_same_thread"
parent_thread_id = f"ckpt-mode-{MODE}"
parent_config = RunnableConfig(configurable={"thread_id": parent_thread_id})

async with AsyncRedisSaver.from_conn_string(REDIS_URL) as ch:
    await ch.asetup()
    chp_list = [t async for t in ch.alist(parent_config)]

print(f"alist returned {len(chp_list)} CheckpointTuple(s) for parent_config={parent_config}\n")
pprint(chp_list[-6], indent=2)
# Per-tuple summary, oldest -> newest
for i, tup in enumerate(reversed(chp_list)):
    cfg = tup.config["configurable"]
    msgs = tup.checkpoint.get("channel_values", {}).get("messages", [])
    print(
        f"[{i}] thread={cfg['thread_id']}  "
        f"ns={cfg.get('checkpoint_ns', '')!r}  "
        f"ckpt_id={cfg['checkpoint_id'][-15:]}  "
        f"parent_ckpt_id={tup.metadata.get('parents', {}).get("", [])[-15:]}  "
        f"step={tup.metadata.get('step'):<3} "
        f"source={tup.metadata.get('source'):<8} "
        f"msgs={len(msgs)}"
    )

# Group by location so it's easy to see which checkpoints belong to parent vs subagent
print("\n--- distinct (thread_id, checkpoint_ns) seen ---")
by_loc: dict[tuple[str, str], int] = {}
for tup in chp_list:
    key = (
            tup.config["configurable"]["thread_id"],
            tup.config["configurable"].get("checkpoint_ns", ""),
            tup.config["configurable"].get("checkpoint_id", ""),
        )
    by_loc[key] = by_loc.get(key, 0) + 1
for (t, n, ch_id), c in sorted(by_loc.items()):
    print(f"  thread={t}  ns={n!r}  ckpt_id={ch_id[-10:]}  count={c}")

# Peek at one full tuple structure
if chp_list:
    head = chp_list[0]
    print("\n--- head CheckpointTuple structure ---")
    print(f"  config.configurable    : {head.config['configurable']}")
    print(f"  checkpoint top-level   : {list(head.checkpoint.keys())}")
    print(f"  metadata               : {dict(head.metadata or {})}")
    print(f"  channel_values keys    : {list(head.checkpoint.get('channel_values', {}).keys())}")
    print(f"  parent_config          : {head.parent_config}")
    print(f"  pending_writes count   : {len(head.pending_writes or [])}")
    if head.pending_writes:
        print(f"  pending_writes[0]      : {head.pending_writes[0]}")


/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/agentic_patterns/subagent_pattern/checkpointer_modes_experiment.py:28: UserWarning: WARNING! cache_control is not default parameter.
                cache_control was transferred to model_kwargs.
                Please confirm that cache_control is what you intended.
  from utils import MODELS, Model
/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/utils/__init__.py:3: UserWarning: WARNING! thinking is not default parameter.
                thinking was transferred to model_kwargs.
                Please confirm that thinking is what you intended.
  from utils.llms import MODELS, Model
/home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/utils/__init__.py:3: UserWarning: WARNING! cache_control is not default parameter.
                cache_control was transferred to model_kwargs.
                Please confirm that cache_control is what you intended.
  from utils.llms import MODELS, Model
/

alist returned 15 CheckpointTuple(s) for parent_config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread'}}

CheckpointTuple(config={'configurable': {'thread_id': 'ckpt-mode-own_same_thread', 'checkpoint_ns': 'tools:7857bbf2-264a-9928-667b-6e8b2f84d7de', 'checkpoint_id': '1f1483ee-e3bb-64ec-bfff-89047ac02233'}}, checkpoint={'type': 'json', 'v': 4, 'ts': '2026-05-05T04:57:34.725847+00:00', 'id': '1f1483ee-e3bb-64ec-bfff-89047ac02233', 'channel_values': {'__start__': {'messages': [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]}}, 'channel_versions': {'__start__': '00000000000000000000000000000001.0.7794106066210351'}, 'versions_seen': {'__input__': {}}, 'updated_channels': ['__start__'], 'pending_sends': []}, metadata={'source': 'input', 'step': -1, 'parents': {'': '1f1483ee-e38a-6c78-8001-6fbd457880a4'}}, parent_config=None, pending_writes=[('5da665bb-9b92-a782-e6ce-8b1cec4db789', 'messages', [{'role': 'user', 'content': 'Tell me one fact about carrots.'}]), ('5

In [3]:
import base64
from typing import Any

import orjson
from redis.asyncio import Redis

from agentic_patterns.subagent_pattern.checkpointer_modes_experiment import REDIS_URL

MODE = "own_same_thread"
thread_id = f"ckpt-mode-{MODE}"


def try_decode_blob(b64: str) -> tuple[bool, str | bytes | None]:
    try:
        raw = base64.b64decode(b64)
    except Exception as e:
        return False, f"base64: {e}"
    try:
        orjson.loads(raw)
    except Exception:
        return False, raw
    return True, raw


def walk_doc(node: Any, path: str, out: list[dict]) -> None:
    if isinstance(node, dict):
        if "blob" in node and isinstance(node["blob"], str):
            ok, payload = try_decode_blob(node["blob"])
            if not ok:
                out.append({"path": f"{path}.blob", "type": node.get("type"),
                            "channel": node.get("channel"), "err_or_raw": payload})
        if "__bytes__" in node and isinstance(node["__bytes__"], str):
            ok, payload = try_decode_blob(node["__bytes__"])
            if not ok:
                out.append({"path": f"{path}.__bytes__", "type": "bytes-marker",
                            "channel": None, "err_or_raw": payload})
        for k, v in node.items():
            walk_doc(v, f"{path}.{k}", out)
    elif isinstance(node, list):
        for i, v in enumerate(node):
            walk_doc(v, f"{path}[{i}]", out)


PREFIXES = ["checkpoint", "checkpoint_write", "checkpoint_latest"]
totals: dict[str, dict[str, int]] = {p: {"docs": 0, "json": 0, "bad": 0} for p in PREFIXES}
all_bad: list[dict] = []
non_json: dict[str, list[str]] = {p: [] for p in PREFIXES}

async with Redis.from_url(REDIS_URL, decode_responses=False) as r:
    for prefix in PREFIXES:
        keys = [k async for k in r.scan_iter(match=f"{prefix}:{thread_id}*".encode())]
        totals[prefix]["docs"] = len(keys)
        for k in keys:
            ktype = (await r.type(k)).decode()
            if ktype != "ReJSON-RL":
                non_json[prefix].append(f"{k.decode()}  (type={ktype})")
                continue
            totals[prefix]["json"] += 1
            doc = await r.json().get(k)
            if not isinstance(doc, (dict, list)):
                continue
            findings: list[dict] = []
            walk_doc(doc, "", findings)
            if findings:
                totals[prefix]["bad"] += 1
                for f in findings:
                    f["key"] = k.decode()
                    f["prefix"] = prefix
                all_bad.extend(findings)

print(f"{'prefix':<22} {'docs':>6} {'json':>6} {'bad':>6}")
for p, t in totals.items():
    print(f"{p:<22} {t['docs']:>6} {t['json']:>6} {t['bad']:>6}")

for p, lst in non_json.items():
    if lst:
        print(f"\nnon-JSON keys under {p}:")
        for s in lst[:5]:
            print(f"  {s}")

print(f"\ntotal bad blobs: {len(all_bad)}\n")
for entry in all_bad[:6]:
    raw = entry["err_or_raw"]
    print(f"BAD {entry['prefix']}{entry['path']}  type={entry.get('type')}  channel={entry.get('channel')}")
    print(f"  key: {entry['key']}")
    if isinstance(raw, bytes):
        print(f"  len: {len(raw)}")
        print(f"  head: {raw[:200]!r}")
        print(f"  tail: {raw[-120:]!r}")
    else:
        print(f"  err: {raw}")
    print()


prefix                   docs   json    bad
checkpoint                 15     15      0
checkpoint_write           24     24      7
checkpoint_latest           3      0      0

non-JSON keys under checkpoint_latest:
  checkpoint_latest:ckpt-mode-own_same_thread:__empty__  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:1cdfbfa5-8898-6599-37d7-3808745e2a99  (type=string)
  checkpoint_latest:ckpt-mode-own_same_thread:tools:7857bbf2-264a-9928-667b-6e8b2f84d7de  (type=string)

total bad blobs: 7

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:tools:7857bbf2-264a-9928-667b-6e8b2f84d7de:1f1483ee-f866-644d-8001-1203735319e0:dd288e45-77a1-3387-be47-b2f875eaf90c:1
  len: 0
  head: b''
  tail: b''

BAD checkpoint_write.blob  type=null  channel=branch:to:model
  key: checkpoint_write:ckpt-mode-own_same_thread:tools:1cdfbfa5-8898-6599-37d7-3808745e2a99:1f1483ee-e398-6725-bfff-0ba1b8c276e0:08ed14ec-d99e-11f5-1821-8e5

In [4]:
async with AsyncRedisSaver.from_conn_string(REDIS_URL) as ch:
    await ch.asetup()
    config = RunnableConfig(configurable={"thread_id": "dbt Agent-1"})
    chp_history = [t async for t in ch.alist(config)]

In [15]:
from langchain_core.messages.utils import count_tokens_approximately
last_tup = chp_history[0]if chp_history else None

chkpt = last_tup.checkpoint if last_tup else None
messages  = chkpt.get("channel_values", {}).get("messages", [])

print(f"last checkpoint has {len(messages)} messages")
approximate_tokens = count_tokens_approximately(messages, chars_per_token=3)
print(f"approximate token count: {approximate_tokens}") 



last checkpoint has 415 messages
approximate token count: 196628


In [ ]:
compaction_messages = messages.copy()
print(f"compaction_messages initially has {len(compaction_messages)} messages")
approximate_tokens = count_tokens_approximately(messages, chars_per_token=3)
print(f"approximate token count before compaction: {approximate_tokens}")

compaction_messages initially has 415 messages
approximate token count before compaction: 196628


In [13]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.messages.utils import count_tokens_approximately

tools_by_type = {
    "write_tools": ["Write", "WriteArtifact"],
    "edit_tools":  ["Edit", "EditArtifact"],
    "read_tools":  ["Read", "ReadArtifact"],
    "misc":        ["Todos", "Skill"],
    "bash":        ["bash", "Grep"],
    "sf_tools":    ["MANAGE_SNOWFLAKE_OBJECTS", "PUBLISH_SEMANTIC_VIEW", "QUERY_SNOWFLAKE", "DEPLOY_CORTEX_AGENT"],
}


def get_message_counts(messages):
    counts = {"human": 0, "ai": 0, "tool": 0}
    for msg in messages:
        if isinstance(msg, HumanMessage):
            counts["human"] += 1
        elif isinstance(msg, AIMessage):
            counts["ai"] += 1
        elif isinstance(msg, ToolMessage):
            counts["tool"] += 1
    return counts


def split_messages_for_type(messages, tool_names):
    """One pass over messages -> (ai_calls, tool_results, remaining) for this tool type.

    ai_calls    : AIMessages whose tool_calls include any name in tool_names
    tool_results: ToolMessages whose tool_call_id matches one of those calls
    remaining   : everything left if we prune both sides
    """
    name_set = set(tool_names)
    matched_call_ids: set[str] = set()
    ai_call_idxs: set[int] = set()

    for i, msg in enumerate(messages):
        if isinstance(msg, AIMessage) and msg.tool_calls:
            if any(c.get("name") in name_set for c in msg.tool_calls):
                ai_call_idxs.add(i)
                for c in msg.tool_calls:
                    if c.get("name") in name_set:
                        matched_call_ids.add(c.get("id"))

    ai_calls, tool_results, remaining = [], [], []
    for i, msg in enumerate(messages):
        if i in ai_call_idxs:
            ai_calls.append(msg)
        elif isinstance(msg, ToolMessage) and msg.tool_call_id in matched_call_ids:
            tool_results.append(msg)
        else:
            remaining.append(msg)
    return ai_calls, tool_results, remaining


def profile_tool_type(messages, tool_type, tool_names, baseline_tokens):
    ai_calls, tool_results, remaining = split_messages_for_type(messages, tool_names)
    call_tok = count_tokens_approximately(ai_calls, chars_per_token=3)
    res_tok  = count_tokens_approximately(tool_results, chars_per_token=3)
    after    = count_tokens_approximately(remaining, chars_per_token=3)
    return {
        "tool_type":   tool_type,
        "calls":       len(ai_calls),
        "call_tok":    call_tok,
        "results":     len(tool_results),
        "res_tok":     res_tok,
        "total_tok":   call_tok + res_tok,
        "after_prune": after,
        "removed":     baseline_tokens - after,
        "msg_counts":  get_message_counts(ai_calls + tool_results),
    }


def profile_all(messages, tools_by_type):
    baseline = count_tokens_approximately(messages, chars_per_token=3)
    baseline_counts = get_message_counts(messages)
    print(f"baseline: {len(messages)} messages, {baseline} tokens, counts={baseline_counts}\n")

    print(
        f"{'tool_type':<14}{'calls':>7}{'call_tok':>10}"
        f"{'results':>9}{'res_tok':>9}{'total_tok':>11}"
        f"{'after_prune':>13}{'removed':>10}  msg_counts"
    )
    print("-" * 95)

    rows = []
    for tool_type, tool_names in tools_by_type.items():
        r = profile_tool_type(messages, tool_type, tool_names, baseline)
        rows.append(r)
        print(
            f"{r['tool_type']:<14}{r['calls']:>7}{r['call_tok']:>10}"
            f"{r['results']:>9}{r['res_tok']:>9}{r['total_tok']:>11}"
            f"{r['after_prune']:>13}{r['removed']:>10}  {r['msg_counts']}"
        )

    # sentinel: drop ALL ToolMessages regardless of type (matches old prune_tool_messages)
    all_tool_msgs = [m for m in messages if isinstance(m, ToolMessage)]
    no_tools      = [m for m in messages if not isinstance(m, ToolMessage)]
    all_tok   = count_tokens_approximately(all_tool_msgs, chars_per_token=3)
    after_all = count_tokens_approximately(no_tools, chars_per_token=3)
    print(
        f"\n{'<all results>':<14}{'-':>7}{'-':>10}"
        f"{len(all_tool_msgs):>9}{all_tok:>9}{all_tok:>11}"
        f"{after_all:>13}{baseline - after_all:>10}  "
        f"{get_message_counts(all_tool_msgs)}"
    )
    return rows


profile = profile_all(messages, tools_by_type)


baseline: 415 messages, 196628 tokens, counts={'human': 12, 'ai': 200, 'tool': 203}

tool_type       calls  call_tok  results  res_tok  total_tok  after_prune   removed  msg_counts
-----------------------------------------------------------------------------------------------
write_tools        30     27566       30     1514      29080       167548     29080  {'human': 0, 'ai': 30, 'tool': 30}
edit_tools         66     18485       66    16093      34578       162050     34578  {'human': 0, 'ai': 66, 'tool': 66}
read_tools         10      2160       15    33744      35904       160724     35904  {'human': 0, 'ai': 10, 'tool': 15}
misc               14      5686       14    24410      30096       166532     30096  {'human': 0, 'ai': 14, 'tool': 14}
bash               46      8944       49    37241      46185       150443     46185  {'human': 0, 'ai': 46, 'tool': 49}
sf_tools           19      2870       24     5109       7979       188649      7979  {'human': 0, 'ai': 19, 'tool': 24}

<a

In [16]:
import pickle
from pathlib import Path


def save_messages(messages, path: str | Path = "experiments/messages_original.pkl") -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(messages, f)
    size = path.stat().st_size
    print(f"saved {len(messages)} messages -> {path.resolve()}  ({size:,} bytes)")
    return path


save_messages(messages, "experiments/messages_196k.pkl")


saved 415 messages -> /home/talhamuammad/Desktop/Personal-Projects/agents-and-workflows/experiments/messages_196k.pkl  (797,552 bytes)


PosixPath('experiments/messages_196k.pkl')